# Project 2: Visual search system

Run `python extract.py` once before this notebook. The notebook imports the code from `extract.py` and `vocab_tree.py` and draws the results. Every figure is also saved in `figures/` for the report.

In [ ]:
import time
import numpy as np, cv2, matplotlib.pyplot as plt
from extract import list_images, object_id, load_gray, load_features
from pathlib import Path

FIG_DIR = Path("figures")

def save(name, photo=False):
    # save the current figure for the report: charts as .pdf + .png, photos as .jpg
    FIG_DIR.mkdir(exist_ok=True)
    fig = plt.gcf()
    if photo:
        fig.savefig(FIG_DIR / f"{name}.jpg", dpi=200, bbox_inches="tight", pil_kwargs={"quality": 90})
    else:
        fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
        fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

plt.rcParams["figure.dpi"] = 90
server_imgs, client_imgs = list_images("server"), list_images("client")
server, client = load_features("server"), load_features("client")
sift = cv2.SIFT_create()

def rgb(path, side=500):
    img = cv2.imread(str(path), cv2.IMREAD_REDUCED_COLOR_4)
    return cv2.cvtColor(cv2.resize(img, (side, side * 3 // 4)), cv2.COLOR_BGR2RGB)

def photos(obj, split="server"):
    return [p for p in (server_imgs if split == "server" else client_imgs) if object_id(p) == obj]

# Section 2: Feature extraction

## 2.1 One object = one building

Database: 3 photos, stored. Query: 1 new photo, searched with. Same building, different moment.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, p in zip(axes, photos(1) + photos(1, "client")):
    ax.imshow(rgb(p)); ax.axis("off")
    ax.set_title(("QUERY  " if "_t" in p.name else "database  ") + p.name,
                 color="C3" if "_t" in p.name else "k")
plt.tight_layout(); save("sec2_one_building", photo=True)

## 2.2 What SIFT keeps

Circles land on window corners, the logo, roof edges. Sky and plain brick get almost nothing: they look the same everywhere.

In [ ]:
gray = load_gray(server_imgs[0])
kps = sift.detect(gray, None)
top = sorted(kps, key=lambda k: -k.response)[:300]
vis = cv2.drawKeypoints(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), top, None, (0, 255, 0),
                        cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(9, 7)); plt.imshow(vis[..., ::-1]); plt.axis("off")
plt.title(f"{server_imgs[0].name}: {len(kps)} keypoints, 300 strongest drawn")
save("sec2_keypoints", photo=True)

## 2.3 Why resize to 1024 px

The task asks for "a few thousand" features per image.

In [ ]:
sizes = [640, 1024, 1600, 2592]
sample = server_imgs[::30]          # 5 images spread over the dataset
n_feat, secs = [], []
for side in sizes:
    n, t0 = [], time.time()
    for p in sample:
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        g = cv2.resize(g, None, fx=side / max(g.shape), fy=side / max(g.shape), interpolation=cv2.INTER_AREA)
        n.append(len(sift.detect(g, None)))
    n_feat.append(np.mean(n)); secs.append((time.time() - t0) / len(sample))

for s, n, t in zip(sizes, n_feat, secs):
    print(f"{s:>5} px   {n:>7.0f} features/image   {t:.2f} s/image")

fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.bar([str(s) for s in sizes], n_feat, color=["0.7", "C0", "0.7", "0.7"])
ax.bar_label(bars, [f"{n:,.0f}\n{t:.2f} s" for n, t in zip(n_feat, secs)])
ax.set_xlabel("long side of image (px)"); ax.set_ylabel("SIFT features per image")
ax.set_ylim(0, max(n_feat) * 1.25); ax.set_title("Chosen: 1024 px")
save("sec2_resolution")

## 2.4 Features per object (report 2a, 2b)

In [ ]:
s_ids, s_cnt = np.unique(server["obj"], return_counts=True)
c_ids, c_cnt = np.unique(client["obj"], return_counts=True)
print(f"server: {s_cnt.mean():,.0f} per object ({len(server['desc']):,} total)")
print(f"client: {c_cnt.mean():,.0f} per object ({len(client['desc']):,} total)")

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.bar(s_ids - 0.2, s_cnt, 0.4, label="server, all photos of the building")
ax.bar(c_ids + 0.2, c_cnt, 0.4, label="client, one query photo")
ax.set_xlabel("object id"); ax.set_ylabel("SIFT features"); ax.set_xticks(s_ids[::2])
ax.set_xlim(0.3, 50.7); ax.legend()
save("sec2_features_per_object")

## 2.5 Matching one query

Match kept only if the best candidate is clearly better than the second (ratio 0.75).

In [ ]:
def good_matches(d_query, d_db, ratio=0.75):
    pairs = cv2.BFMatcher(cv2.NORM_L2).knnMatch(d_query, d_db, k=2)
    return [m for m, n in pairs if m.distance < ratio * n.distance]

q = photos(13, "client")[0]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, db in zip(axes, [photos(13)[0], photos(30)[0]]):
    g1, g2 = load_gray(q), load_gray(db)
    k1, d1 = sift.detectAndCompute(g1, None)
    k2, d2 = sift.detectAndCompute(g2, None)
    good = good_matches(d1, d2)
    ax.imshow(cv2.drawMatches(g1, k1, g2, k2, good[:80], None, flags=2)); ax.axis("off")
    ax.set_title(f"{q.name} vs {db.name}: {len(good)} matches")
plt.tight_layout(); save("sec2_matches", photo=True)

## 2.6 One query against all 50 buildings

Brute force: compare every query feature with every database feature. This is the answer the vocabulary tree must reproduce, much faster.

In [ ]:
QUERY = 13
dq = client["desc"][client["obj"] == QUERY].astype(np.float32)
t0 = time.time()
scores = [len(good_matches(dq, server["desc"][server["obj"] == o].astype(np.float32))) for o in s_ids]
secs = time.time() - t0
ranking = s_ids[np.argsort(scores)[::-1]]

print(f"top-5: {[int(o) for o in ranking[:5]]}   correct: {QUERY}")
print(f"{len(dq):,} x {len(server['desc']):,} = {len(dq) * len(server['desc']):,} distances, {secs:.0f} s")

fig, ax = plt.subplots(figsize=(14, 3.2))
ax.bar(s_ids, scores, color=["C3" if o == QUERY else "0.6" for o in s_ids])
ax.set_xlabel("database object id"); ax.set_ylabel("good matches")
ax.set_xticks(s_ids[::2]); ax.set_xlim(0.3, 50.7)
ax.set_title(f"Query obj{QUERY}_t1 vs every building")
save("sec2_query_vs_all")

# Section 3: Vocabulary tree

Each node stores its cluster center (3a). Each leaf stores how many features of each object landed there, plus `idf = ln(N / N_i)` (3c). Building all three trees takes about 45 s.

In [ ]:
from vocab_tree import hi_kmeans, add_database, assign_leaves

trees = {}
for b, depth in [(4, 3), (4, 5), (5, 7)]:
    t0 = time.time()
    tree = hi_kmeans(server["desc"], b, depth)
    add_database(tree, server["desc"], server["obj"])
    trees[(b, depth)] = tree
    print(f"b={b} depth={depth}: {tree.n_leaves:>6,} leaves, {time.time() - t0:4.0f} s, "
          f"{len(server['desc']) / tree.n_leaves:8,.1f} features per leaf")

Distances per query descriptor: the tree compares with b centers per level, a flat vocabulary with every word.

In [ ]:
labels = [f"b={b}, d={d}" for b, d in trees]
tree_cost = [b * d for b, d in trees]
flat_cost = [b ** d for b, d in trees]
for l, t, f in zip(labels, tree_cost, flat_cost):
    print(f"{l}:  tree {t:>3}   flat {f:>6,}   {f / t:>6,.0f}x fewer")

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(x - 0.2, flat_cost, 0.4, label="flat vocabulary", color="0.6")
ax.bar(x + 0.2, tree_cost, 0.4, label="vocabulary tree", color="C0")
ax.set_yscale("log"); ax.set_xticks(x, labels)
ax.set_ylabel("distances per descriptor (log)"); ax.legend()
save("sec3_distances")

# Section 4: Querying

To do. `trees[(b, depth)]` holds the three trees, `client` the query features, `assign_leaves(tree, desc)` gives the leaf of each query feature.